# Lesson 07: Image Registration — Python (Colab)

This notebook walks through a synthetic feature-based registration pipeline: correspondences, robust affine estimation, inverse warping, and overlay evaluation.

**How to use:** Run all cells, then add outliers or swap in real feature matches/homography estimation.

## Setup

This notebook uses NumPy and Matplotlib (preinstalled in Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Target and moving (synthetic) images
H,W=120,160
yy,xx=np.mgrid[0:H,0:W]
def draw_shape(shift=(0,0), angle=0, scale=1.0):
    img=np.zeros((H,W),np.float32)
    # simple shapes
    img[30:80,40:110]=120
    img[50:65,60:130]=200
    img[(yy-90)**2 + (xx-60)**2 < 12*12]=180
    # apply transform via inverse sampling
    a=np.deg2rad(angle); c,s=np.cos(a),np.sin(a)
    M=np.array([[scale*c,-scale*s,shift[0]],[scale*s,scale*c,shift[1]],[0,0,1]])
    Minv=np.linalg.inv(M)
    Xd,Yd=np.mgrid[0:W,0:H]
    ones=np.ones_like(Xd)
    src=Minv @ np.stack([Xd,Yd,ones],0).reshape(3,-1)
    xs,ys=src[0]/src[2], src[1]/src[2]
    x0=np.clip(np.floor(xs).astype(int),0,W-1); x1=np.clip(x0+1,0,W-1)
    y0=np.clip(np.floor(ys).astype(int),0,H-1); y1=np.clip(y0+1,0,H-1)
    wx=xs-x0; wy=ys-y0
    Ia=img[y0,x0]; Ib=img[y0,x1]; Ic=img[y1,x0]; Id=img[y1,x1]
    out=((1-wx)*(1-wy)*Ia + wx*(1-wy)*Ib + (1-wx)*wy*Ic + wx*wy*Id).reshape(H,W)
    mask=((xs>=0)&(xs<W-1)&(ys>=0)&(ys<H-1)).reshape(H,W)
    return out, mask, M

target, _, _ = draw_shape((0,0),0,1.0)
# moving is a transformed target
moving, mask_m, M_true = draw_shape((8,5),12,1.05)

# Synthetic correspondences (true + outliers)
src_pts = np.array([[45,35],[100,35],[100,70],[55,75],[60,90],[30,80]],np.float32)
ones=np.ones((len(src_pts),1),np.float32)
tgt_true = (M_true @ np.c_[src_pts,ones].T).T
tgt_pts = (tgt_true[:,:2]/tgt_true[:,2:3]).astype(np.float32)
tgt_pts = tgt_pts + np.random.default_rng(1).normal(0,0.6,tgt_pts.shape).astype(np.float32)
# add outliers
tgt_pts_noisy = tgt_pts.copy(); tgt_pts_noisy[-1] += np.array([25,-20]); tgt_pts_noisy[-2] += np.array([-20,15])

def estimate_affine(src,dst):
    X = np.c_[src, np.ones(len(src))]
    A,_,_,_ = np.linalg.lstsq(X, dst, rcond=None)
    M = np.eye(3); M[:2,:]=A.T; return M

def ransac_affine(src,dst,iters=200,thresh=3.0):
    best=None; best_inl=None
    rng=np.random.default_rng(0)
    for _ in range(iters):
        idx=rng.choice(len(src),3,replace=False)
        M=estimate_affine(src[idx],dst[idx])
        proj=(M @ np.c_[src,np.ones(len(src))].T).T; proj=proj[:,:2]/proj[:,2:3]
        err=np.linalg.norm(proj-dst,axis=1)
        inl=err<thresh
        if best_inl is None or inl.sum()>best_inl.sum(): best,best_inl=M,inl
    if best_inl.sum()>=3: best=estimate_affine(src[best_inl],dst[best_inl])
    return best,best_inl

M_est, inl = ransac_affine(src_pts, tgt_pts_noisy)
Minv = np.linalg.inv(M_est)
Xd,Yd=np.mgrid[0:W,0:H]
src=(Minv @ np.stack([Xd,Yd,np.ones_like(Xd)],0).reshape(3,-1))
xs,ys=src[0]/src[2], src[1]/src[2]
x0=np.clip(np.floor(xs).astype(int),0,W-1); x1=np.clip(x0+1,0,W-1)
y0=np.clip(np.floor(ys).astype(int),0,H-1); y1=np.clip(y0+1,0,H-1)
wx=xs-x0; wy=ys-y0
Ia=moving[y0,x0]; Ib=moving[y0,x1]; Ic=moving[y1,x0]; Id=moving[y1,x1]
warped=((1-wx)*(1-wy)*Ia + wx*(1-wy)*Ib + (1-wx)*wy*Ic + wx*wy*Id).reshape(H,W)
valid=((xs>=0)&(xs<W-1)&(ys>=0)&(ys<H-1)).reshape(H,W)
overlay = 0.5*target + 0.5*np.where(valid,warped,0)

fig,axs=plt.subplots(1,4,figsize=(14,3.5))
axs[0].imshow(target,cmap='gray'); axs[0].scatter(tgt_pts_noisy[:,0],tgt_pts_noisy[:,1],c='cyan',s=20); axs[0].set_title('Target + matches')
axs[1].imshow(moving,cmap='gray'); axs[1].scatter(src_pts[:,0],src_pts[:,1],c='orange',s=20); axs[1].set_title('Moving')
axs[2].imshow(warped,cmap='gray'); axs[2].set_title('Warped moving')
axs[3].imshow(overlay,cmap='gray'); axs[3].set_title('Overlay')
for a in axs: a.axis('off')
plt.tight_layout(); plt.show()
print('Inliers:', inl.sum(), '/', len(inl))
print('Estimated M:\n', M_est)

## Experiments

replace synthetic points with real features (<code>cv2.ORB_create</code>/<code>BFMatcher</code>), estimate a homography with <code>cv2.findHomography(..., cv2.RANSAC)</code>, and compute NCC/SSD in the overlap.